# Fish Audio S2 Pro Colab Experiment

This notebook is for learning Fish Speech's S2 Pro TTS / voice-clone flow on a Colab A100/H100 runtime.

- This repo owns the helper code and artifacts.
- `vendor/fish-speech` is the upstream reference implementation.
- HF model files stay in ephemeral `/content`; Google Drive stores only our generated outputs, logs, manifests, and optional reference copies.
- Use only voices/audio you are authorized to clone or synthesize.

In [ ]:
#@title 0. Runtime configuration
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys, time

PUBLIC_REPO_URL = "https://github.com/liuwen/qwen-asr-eval.git"  #@param {type:"string"}
PUBLIC_REPO_BRANCH = "exp/fishaudio"  #@param {type:"string"}
REPO_DIR = Path("/content/qwen-asr-eval")
EXPERIMENT_DIR = REPO_DIR / "experiments" / "fishaudio_s2_pro"
WORK_ROOT = Path("/content/fishaudio_s2_pro")
DRIVE_ROOT = Path("/content/drive/MyDrive/voice/fishaudio-s2-pro")
RUN_ID = ""  #@param {type:"string"}
MOUNT_DRIVE = True  #@param {type:"boolean"}

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This experiment targets Python 3.12; current kernel is {sys.version}")

def run(cmd, *, cwd=None, env=None, check=True):
    started = time.time()
    display_cwd = str(cwd) if cwd else os.getcwd()
    print(f"\n[{time.strftime('%H:%M:%S')}] cwd={display_cwd}", flush=True)
    print("$", " ".join(map(str, cmd)), flush=True)
    proc_env = os.environ.copy()
    if env:
        proc_env.update(env)
    proc_env["PYTHONUNBUFFERED"] = "1"
    proc_env.setdefault("PIP_PROGRESS_BAR", "on")
    proc_env.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
    result = subprocess.run(cmd, cwd=cwd, env=proc_env, check=check)
    print(f"[{time.strftime('%H:%M:%S')}] done in {time.time() - started:.1f}s", flush=True)
    return result

print("Python:", sys.version)
print("Repo target:", PUBLIC_REPO_URL, PUBLIC_REPO_BRANCH)


In [ ]:
#@title 1. Mount Drive, clone this repo, and initialize submodules
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if REPO_DIR.exists():
    run(["git", "fetch", "origin", PUBLIC_REPO_BRANCH], cwd=REPO_DIR, check=False)
    run(["git", "checkout", PUBLIC_REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", PUBLIC_REPO_BRANCH], cwd=REPO_DIR, check=False)
else:
    run([
        "git", "clone", "--recurse-submodules", "--branch", PUBLIC_REPO_BRANCH,
        PUBLIC_REPO_URL, str(REPO_DIR),
    ])

run(["git", "submodule", "update", "--init", "--recursive"], cwd=REPO_DIR)
FISH_SPEECH_DIR = EXPERIMENT_DIR / "vendor" / "fish-speech"
if not FISH_SPEECH_DIR.exists():
    raise RuntimeError(
        f"Fish Speech submodule not found at {FISH_SPEECH_DIR}. "
        "Re-run this cell after confirming the branch contains experiments/fishaudio_s2_pro/vendor/fish-speech."
    )
print("Experiment dir:", EXPERIMENT_DIR)
print("Fish Speech submodule:", FISH_SPEECH_DIR)


In [ ]:
#@title 2. Install system tools and experiment helper
run(["apt-get", "-qq", "update"])
run(["apt-get", "-y", "-qq", "install", "ffmpeg", "portaudio19-dev", "libsox-dev"])
run([sys.executable, "-m", "pip", "install", "-U", "uv"])

if not EXPERIMENT_DIR.exists():
    raise RuntimeError(f"Experiment directory not found: {EXPERIMENT_DIR}")

# Local development surface for our helper code. Do not install the repo-root
# requirements-colab.txt here; that file belongs to the Qwen ASR workflow.
print("Installing/syncing the local Fish Audio helper package with uv...", flush=True)
run(["uv", "--color", "always", "sync", "--python", "3.12"], cwd=EXPERIMENT_DIR)
run([sys.executable, "-m", "pip", "install", "-e", str(EXPERIMENT_DIR)])


In [ ]:
#@title 3. Create run paths, check GPU, and read HF_TOKEN
HELPER_SRC = EXPERIMENT_DIR / "src"
if HELPER_SRC.exists() and str(HELPER_SRC) not in sys.path:
    sys.path.insert(0, str(HELPER_SRC))

from fishaudio_s2_pro import ExperimentPaths, read_colab_secret, require_minimum_vram, resolve_fish_uv_extra
from fishaudio_s2_pro.artifacts import write_json

paths = ExperimentPaths.for_colab(
    project_root=EXPERIMENT_DIR,
    work_root=WORK_ROOT,
    drive_root=DRIVE_ROOT,
    run_id=RUN_ID.strip() or None,
).ensure()
if not paths.fish_repo.exists():
    raise RuntimeError(f"Fish Speech submodule not found at {paths.fish_repo}")

gpus = require_minimum_vram(24)
FISH_UV_EXTRA_REQUEST = "auto"  #@param ["auto", "cu126", "cu128", "cu129"]
FISH_UV_EXTRA = resolve_fish_uv_extra(FISH_UV_EXTRA_REQUEST)

HF_TOKEN = read_colab_secret("HF_TOKEN", required=True)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.update(paths.hf_env())

write_json(paths.manifests_dir / "paths.json", paths.manifest())
write_json(paths.manifests_dir / "runtime.json", {"gpus": gpus, "fish_uv_extra": FISH_UV_EXTRA})

print("Run dir:", paths.run_dir)
print("Ephemeral checkpoint dir:", paths.checkpoint_dir)
print("GPU inventory:", gpus)
print("Fish uv extra:", FISH_UV_EXTRA)


In [ ]:
#@title 4. Install Fish Speech upstream environment with uv
fish_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=paths.fish_repo, text=True).strip()
print("Fish Speech commit:", fish_commit)
write_json(paths.manifests_dir / "fish_speech_commit.json", {"commit": fish_commit})

print("Installing Fish Speech upstream dependencies with uv. A fresh Colab VM can take several minutes.", flush=True)
run(["uv", "--color", "always", "sync", "--python", "3.12", "--extra", FISH_UV_EXTRA], cwd=paths.fish_repo)


In [ ]:
#@title 5. Download Fish Audio S2 Pro weights to ephemeral /content
MODEL_ID = "fishaudio/s2-pro"
paths.checkpoint_dir.mkdir(parents=True, exist_ok=True)

HF_DOWNLOAD_WORKERS = 2  #@param {type:"integer"}
download_log_path = paths.logs_dir / "hf_download.log"

if not (paths.checkpoint_dir / "codec.pth").exists():
    print("Installing Hugging Face CLI in the notebook kernel...", flush=True)
    run([sys.executable, "-m", "pip", "install", "-U", "huggingface_hub"])
    if shutil.which("hf") is None:
        raise RuntimeError("`hf` CLI was not found after installing huggingface_hub")

    print("Downloading S2 Pro weights to ephemeral /content.", flush=True)
    print("Download log:", download_log_path, flush=True)
    download_env = os.environ.copy()
    download_env["HF_TOKEN"] = HF_TOKEN
    download_env["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    download_env.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
    hf_cmd = [
        "hf", "download", MODEL_ID,
        "--local-dir", str(paths.checkpoint_dir),
        "--max-workers", str(max(1, int(HF_DOWNLOAD_WORKERS))),
    ]
    shell_cmd = " ".join(shlex.quote(str(part)) for part in hf_cmd)
    shell_cmd = f"set -o pipefail; {shell_cmd} 2>&1 | tee -a {shlex.quote(str(download_log_path))}"
    run(["bash", "-lc", shell_cmd], cwd=paths.fish_repo, env=download_env)
else:
    print("Checkpoint already present in ephemeral /content:", paths.checkpoint_dir)

write_json(paths.manifests_dir / "model_download.json", {"model_id": MODEL_ID, "checkpoint_dir": str(paths.checkpoint_dir), "storage": "ephemeral_content"})


In [ ]:
#@title 6. Start upstream API server and run health check
from fishaudio_s2_pro import start_api_server, wait_for_http

API_PORT = 8080
COMPILE_MODEL = False  #@param {type:"boolean"}
USE_HALF = False  #@param {type:"boolean"}

api_proc = start_api_server(paths, port=API_PORT, compile_model=COMPILE_MODEL, half=USE_HALF)
print("API PID:", api_proc.pid)
print("API log:", api_proc.log_path)
wait_for_http(f"http://127.0.0.1:{API_PORT}/v1/health", timeout_seconds=900)
print("API healthy:", f"http://127.0.0.1:{API_PORT}/v1/health")


In [ ]:
#@title 7. No-reference TTS smoke
from IPython.display import Audio, display
from fishaudio_s2_pro import post_tts

NO_REF_TEXT = "Hello from Fish Audio S2 Pro running on a Colab GPU."  #@param {type:"string"}
no_ref_output = paths.outputs_dir / "no_reference_smoke.wav"
no_ref_manifest = post_tts(
    base_url=f"http://127.0.0.1:{API_PORT}",
    text=NO_REF_TEXT,
    output_path=no_ref_output,
    reference_id=None,
    seed=42,
)
print(no_ref_manifest)
display(Audio(filename=str(no_ref_output)))


In [ ]:
#@title 8. Prepare optional reference voice
from fishaudio_s2_pro import prepare_reference_voice, sanitize_voice_id

REFERENCE_AUDIO_PATH = ""  #@param {type:"string"}
REFERENCE_TEXT = ""  #@param {type:"string"}
VOICE_ID = "demo_voice"  #@param {type:"string"}
REFERENCE_START_SECONDS = 0.0  #@param {type:"number"}
REFERENCE_MAX_SECONDS = 10.0  #@param {type:"number"}
SAVE_REFERENCE_COPY_TO_DRIVE = False  #@param {type:"boolean"}

voice_id = sanitize_voice_id(VOICE_ID)
reference_manifest = None
if REFERENCE_AUDIO_PATH.strip():
    reference_manifest = prepare_reference_voice(
        source_audio=REFERENCE_AUDIO_PATH.strip(),
        reference_text=REFERENCE_TEXT,
        voice_id=voice_id,
        fish_repo=paths.fish_repo,
        run_dir=paths.run_dir,
        start_seconds=REFERENCE_START_SECONDS,
        max_seconds=REFERENCE_MAX_SECONDS,
        save_reference_copy_to_drive=SAVE_REFERENCE_COPY_TO_DRIVE,
    )
    print(reference_manifest)
else:
    print("Set REFERENCE_AUDIO_PATH and REFERENCE_TEXT to run voice cloning.")


In [ ]:
#@title 9. Reference voice TTS smoke
REFERENCE_TTS_TEXT = "This is a short reference voice smoke test."  #@param {type:"string"}

if reference_manifest is None:
    print("Skipping reference TTS because no reference voice was prepared.")
else:
    ref_output = paths.outputs_dir / f"reference_{voice_id}_smoke.wav"
    ref_manifest = post_tts(
        base_url=f"http://127.0.0.1:{API_PORT}",
        text=REFERENCE_TTS_TEXT,
        output_path=ref_output,
        reference_id=voice_id,
        seed=42,
    )
    print(ref_manifest)
    display(Audio(filename=str(ref_output)))


In [ ]:
#@title 10. Launch upstream web demo after API smoke
from fishaudio_s2_pro import start_gradio_webui, stop_process

WEB_DEMO_MODE = "gradio"  #@param ["none", "gradio", "awesome"]
STOP_API_BEFORE_WEBUI = True  #@param {type:"boolean"}
WEB_PORT = 7860
active_web_url = None
web_proc = None

if WEB_DEMO_MODE != "none" and STOP_API_BEFORE_WEBUI and "api_proc" in globals():
    stop_process(api_proc)
    print("Stopped API smoke process before launching web demo.")

if WEB_DEMO_MODE == "gradio":
    WEB_PORT = 7860
    web_proc = start_gradio_webui(paths, compile_model=COMPILE_MODEL, half=USE_HALF)
    active_web_url = f"http://127.0.0.1:{WEB_PORT}"
    wait_for_http(active_web_url, timeout_seconds=900)
elif WEB_DEMO_MODE == "awesome":
    WEB_PORT = 8888
    run(["npm", "install"], cwd=paths.fish_repo / "awesome_webui")
    run(["npm", "run", "build"], cwd=paths.fish_repo / "awesome_webui")
    web_proc = start_api_server(paths, port=WEB_PORT, compile_model=COMPILE_MODEL, half=USE_HALF)
    wait_for_http(f"http://127.0.0.1:{WEB_PORT}/v1/health", timeout_seconds=900)
    active_web_url = f"http://127.0.0.1:{WEB_PORT}/ui"
    wait_for_http(active_web_url, timeout_seconds=120)

if active_web_url:
    print("Web demo ready:", active_web_url)
    print("Web log:", web_proc.log_path)
else:
    print("Web demo not started.")


In [ ]:
#@title 11. Optional Cloudflare named tunnel
from fishaudio_s2_pro import start_cloudflared_named_tunnel

START_CLOUDFLARE_TUNNEL = False  #@param {type:"boolean"}
CLOUDFLARED_CUSTOM_DOMAIN = ""  #@param {type:"string"}

tunnel_proc = None
if START_CLOUDFLARE_TUNNEL:
    if not active_web_url:
        raise RuntimeError("Start a web demo before launching the tunnel.")
    if shutil.which("cloudflared") is None:
        Path("/content/bin").mkdir(parents=True, exist_ok=True)
        run([
            "curl", "-L", "--fail", "--output", "/content/bin/cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ])
        run(["chmod", "+x", "/content/bin/cloudflared"])
        os.environ["PATH"] = "/content/bin:" + os.environ["PATH"]
    tunnel_token = read_colab_secret("CLOUDFLARED_TUNNEL_TOKEN", required=True)
    tunnel_proc = start_cloudflared_named_tunnel(
        token=tunnel_token,
        cwd=paths.work_root,
        log_path=paths.logs_dir / "cloudflared.log",
    )
    print("Cloudflare named tunnel started. PID:", tunnel_proc.pid)
    print("Tunnel log:", tunnel_proc.log_path)
    if CLOUDFLARED_CUSTOM_DOMAIN.strip():
        print("Expected public URL:", CLOUDFLARED_CUSTOM_DOMAIN.strip())
else:
    print("Tunnel skipped.")


In [ ]:
#@title 12. Inspect logs and final artifact manifest
from fishaudio_s2_pro.artifacts import read_tail

manifest = {
    "run_id": paths.run_id,
    "run_dir": str(paths.run_dir),
    "outputs_dir": str(paths.outputs_dir),
    "logs_dir": str(paths.logs_dir),
    "active_web_url": active_web_url,
    "web_demo_mode": WEB_DEMO_MODE,
}
write_json(paths.manifests_dir / "final_manifest.json", manifest)
print(json.dumps(manifest, indent=2))

for log_path in sorted(paths.logs_dir.glob("*.log")):
    print("\n====", log_path, "====")
    print(read_tail(log_path, lines=40))
